<a href="https://colab.research.google.com/github/anjalidrj/GO-analysis-of-peptides/blob/main/20260708_impakt_photo_stress_peptides.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==================== CELL 1 — setup + upload ====================
!pip -q install goatools >/dev/null 2>&1

import os, re, glob, gzip, urllib.request
import pandas as pd
from collections import defaultdict

# --- settings you might tweak ---
MIN_CROPS    = 4              # "common" = pathway hit in >= this many crops
PHOTO_ROOT   = "GO:0015979"   # photosynthesis
STRESS_ROOT  = "GO:0006950"   # response to stress
COTTON_TAXID = "3635"         # Gossypium hirsutum (native gene2go)

from google.colab import files
print("Upload all 9 files: 5 exact_hits_final_*.csv + 4 gProfiler_*_athaliana_*.csv")
files.upload()

def find(pattern):
    hits = glob.glob(pattern)
    return hits[0] if hits else None

CROP_FILES = {
    "soy":    find("exact_hits_final_soy.csv"),
    "chilli": find("exact_hits_final_ca.csv"),
    "cotton": find("exact_hits_final_cotton.csv"),
    "grape":  find("exact_hits_final_grape.csv"),
    "tomato": find("exact_hits_final_sol.csv"),
}
ORTH_FILES = {
    "soy":    find("gProfiler_gmax_athaliana.csv"),
    "chilli": find("gProfiler_cannuum_athaliana.csv"),
    "grape":  find("gProfiler_vvinifera_athaliana.csv"),
    "tomato": find("gProfiler_slycopersicum_athaliana.csv"),
}
print("\nCrop hit tables found:")
for k, v in CROP_FILES.items(): print(f"  {k:7s}: {v}")
print("Ortholog maps found:")
for k, v in ORTH_FILES.items(): print(f"  {k:7s}: {v}")

In [ ]:
import os
if os.path.exists("go-basic.obo") and os.path.getsize("go-basic.obo") < 10_000_000:
    os.remove("go-basic.obo")   # remove truncated download
    print("removed partial file")

In [ ]:
# ==================== CELL 2 — GO ontology + pathway subtrees ====================
from goatools.obo_parser import GODag

# download the ontology if not already present (direct GO mirror, with progress)
import os
if not os.path.exists("go-basic.obo") or os.path.getsize("go-basic.obo") < 10_000_000:
    !wget -c -O go-basic.obo "https://release.geneontology.org/2024-01-17/ontology/go-basic.obo"

# load WITH relationships so part_of links (needed for photosynthesis) are included
godag = GODag("go-basic.obo", optional_attrs={"relationship"})

def subtree(root):
    """All descendants of root via is_a + part_of, walking downward."""
    seen = set()
    stack = [root]
    while stack:
        cur = stack.pop()
        if cur in seen:
            continue
        seen.add(cur)
        node = godag.get(cur)
        if not node:
            continue
        # is_a children
        for ch in node.children:
            stack.append(ch.id)
        # relationship children (part_of etc.) via reverse links
        for rel_set in getattr(node, "relationship_rev", {}).values():
            for ch in rel_set:
                stack.append(ch.id)
    return seen

photo_terms  = subtree(PHOTO_ROOT)
stress_terms = subtree(STRESS_ROOT)
print(f"photosynthesis subtree : {len(photo_terms)} terms")
print(f"stress subtree         : {len(stress_terms)} terms")

In [ ]:
# ==================== CELL 3 — Arabidopsis AT-locus -> GO ====================
ARAB_GAF = "tair.gaf.gz"
if not os.path.exists(ARAB_GAF):
    !wget -c -O tair.gaf.gz "https://current.geneontology.org/annotations/tair.gaf.gz"

at_go = defaultdict(set)
with gzip.open(ARAB_GAF, "rt") as fh:
    for line in fh:
        if line.startswith("!"):
            continue
        c = line.rstrip("\n").split("\t")
        if len(c) < 11:
            continue
        go = c[4]
        # AT-locus can appear in object id (c[1]), symbol (c[2]) or synonyms (c[10])
        for token in [c[1], c[2], *c[10].split("|")]:
            if re.match(r"AT[1-5MC]G\d{5}", token, re.I):
                at_go[token.upper()].add(go)

print(f"Arabidopsis AT loci with GO: {len(at_go)}")
ex = next(iter(at_go))
print("example:", ex, "->", list(at_go[ex])[:3])

In [ ]:
# ==================== CELL 4 — cotton LOC -> GO (native NCBI) ====================
if not os.path.exists("gene2go.gz"):
    print("Downloading NCBI gene2go (~hundreds of MB, 1-3 min)...")
    !wget -c -O gene2go.gz "https://ftp.ncbi.nlm.nih.gov/gene/DATA/gene2go.gz"

cotton_go = defaultdict(set)
with gzip.open("gene2go.gz", "rt") as fh:
    for line in fh:
        if line.startswith("#"):
            continue
        p = line.rstrip("\n").split("\t")
        if p[0] != COTTON_TAXID:      # 3635 = G. hirsutum
            continue
        cotton_go["LOC" + p[1]].add(p[2])

print(f"Cotton LOC genes with GO: {len(cotton_go)}")
ex = next(iter(cotton_go))
print("example:", ex, "->", list(cotton_go[ex])[:3])

In [ ]:
# ==================== CELL 5 — load ortholog maps: crop gene -> {AT} ====================
def load_orth(path):
    m = defaultdict(set)
    if not path or not os.path.exists(path):
        return m
    df = pd.read_csv(path)
    for a, at in zip(df["initial_alias"], df["ortholog_ensg"]):
        at = str(at)
        if at.startswith("AT"):
            m[str(a).upper()].add(at.upper())
    return m

orth = {c: load_orth(p) for c, p in ORTH_FILES.items()}
for c, m in orth.items():
    print(f"{c:7s}: {len(m)} genes with an AT ortholog")

In [ ]:
# ==================== CELL 6 — per-crop gene_id extractor ====================
def clean_gene(crop, row):
    gd  = str(row.get("gene_description", ""))
    gid = str(row.get("gene_id", ""))
    if crop == "cotton":
        m = re.search(r"\((LOC\d+)\)", gd)
        return m.group(1) if m else None
    if crop == "tomato":
        m = re.search(r"gene:gene-(\S+)", gd)
        return (m.group(1) if m else gid.replace("gene-", "")).upper()
    if crop == "grape":
        m = re.search(r"gene:(\S+)", gd)
        g = (m.group(1) if m else gid).upper()
        return g.replace("VITIS", "VITVI", 1)   # match the Vitvi ortholog keys
    return gid.upper()   # soy (GLYMA_..), chilli (T459_..)

# sanity: show one extracted gene per crop
for crop, path in CROP_FILES.items():
    if not path:
        print(f"{crop:7s}: NO FILE"); continue
    r0 = pd.read_csv(path, nrows=1).iloc[0]
    print(f"{crop:7s}: {clean_gene(crop, r0)}")

In [ ]:
# verify extracted IDs actually match the map keys
for crop, path in CROP_FILES.items():
    if not path:
        continue
    df = pd.read_csv(path)
    genes = {clean_gene(crop, r) for _, r in df.iterrows() if clean_gene(crop, r)}
    if crop == "cotton":
        matched = sum(1 for g in genes if g in cotton_go)
    else:
        matched = sum(1 for g in genes if g in orth[crop])
    print(f"{crop:7s}: {len(genes):5d} unique genes | {matched:5d} matched in map "
          f"({100*matched/max(len(genes),1):.0f}%)")

In [ ]:
tdf = pd.read_csv(ORTH_FILES["tomato"])
map_keys = set(tdf["initial_alias"].astype(str).str.upper())
map_keys_nover = {k.split(".")[0] for k in map_keys}
hdf = pd.read_csv(CROP_FILES["tomato"])
allh = {clean_gene("tomato", r) for _, r in hdf.iterrows()}
m_ver   = sum(1 for g in allh if g in map_keys)
m_nover = sum(1 for g in allh if g.split(".")[0] in map_keys_nover)
print(f"map size: {len(map_keys)} | match WITH version: {m_ver} | match WITHOUT version: {m_nover}")

In [ ]:
# ==================== CELL 7 — tag peptides by pathway, per crop ====================
results = {"photo": defaultdict(lambda: defaultdict(set)),
          "stress": defaultdict(lambda: defaultdict(set))}

def go_for_gene(crop, gene):
    if crop == "cotton":
        return cotton_go.get(gene, set())
    s = set()
    for at in orth[crop].get(gene, ()):
        s |= at_go.get(at, set())
    return s

for crop, path in CROP_FILES.items():
    if not path:
        continue
    df = pd.read_csv(path)
    for _, row in df.iterrows():
        gene = clean_gene(crop, row)
        if not gene:
            continue
        gs = go_for_gene(crop, gene)
        if not gs:
            continue
        pep = str(row["peptide_id"])
        if gs & photo_terms:
            results["photo"][crop][pep].add(gene)
        if gs & stress_terms:
            results["stress"][crop][pep].add(gene)
    print(f"{crop:7s}: {len(results['photo'][crop]):4d} peptides->photo | "
          f"{len(results['stress'][crop]):4d} peptides->stress")

In [ ]:
# ==================== CELL 8 — intersect across crops + save (3+ crops, with seq) ====================
# build peptide_id -> sequence lookup from all crop tables
pep_seq = {}
for crop, path in CROP_FILES.items():
    if not path:
        continue
    d = pd.read_csv(path, usecols=lambda c: c in ("peptide_id", "peptide_seq"))
    for pid, seq in zip(d["peptide_id"].astype(str), d["peptide_seq"].astype(str)):
        pep_seq.setdefault(pid, seq)

MIN_CROPS_OUT = 3          # keep peptides hitting the pathway in >= this many crops

def summarize(pathway):
    per = results[pathway]
    crops = [c for c in CROP_FILES if per.get(c)]
    pep_crops, pep_genes = defaultdict(set), defaultdict(dict)
    for crop in crops:
        for pep, genes in per[crop].items():
            pep_crops[pep].add(crop)
            pep_genes[pep][crop] = ";".join(sorted(genes))
    rows = []
    for pep, cset in pep_crops.items():
        if len(cset) >= MIN_CROPS_OUT:
            row = {"peptide_id": pep,
                   "peptide_seq": pep_seq.get(pep, ""),
                   "n_crops": len(cset)}
            for crop in CROP_FILES:
                row[crop] = pep_genes[pep].get(crop, "")
            rows.append(row)
    return pd.DataFrame(rows).sort_values("n_crops", ascending=False)

photo_out  = summarize("photo")
stress_out = summarize("stress")
photo_out.to_csv("common_peptides_photosynthesis.csv", index=False)
stress_out.to_csv("common_peptides_stress.csv", index=False)

def report(name, out):
    print(f"\n=== {name}: {len(out)} peptides in >= {MIN_CROPS_OUT} crops ===")
    for n in (5, 4, 3):
        print(f"  {n} crops: {(out['n_crops'] == n).sum()}")
    print(out.head(20).to_string(index=False))

report("PHOTOSYNTHESIS", photo_out)
report("STRESS", stress_out)

files.download("common_peptides_photosynthesis.csv")
files.download("common_peptides_stress.csv")